# 01 · Entendimiento del Negocio
### Predicción de Riesgo de Incumplimiento Crediticio · Dataset UCI

---
> **Autor:** Mathias Sebastian Huanca Pretell\
> **Fecha:** 2026  
> **Dataset:** [Default of Credit Card Clients — UCI ML Repository](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients)  
> **Notebook:** `01_business_understanding.ipynb`  
> **Estado:** ✅ Completo

---

**Hoja de ruta del proyecto**

| # | Notebook | Estado |
|---|----------|--------|
| 01 | Entendimiento del Negocio | ✅ Actual |
| 02 | Análisis Exploratorio de Datos (EDA) | 🔜 |
| 03 | Preprocesamiento y Feature Engineering | 🔜 |
| 04 | Modelado y Evaluación | 🔜 |
| 05 | Conclusiones y Recomendaciones de Negocio | 🔜 |

## Tabla de Contenidos

1. [Contexto del Problema](#1-contexto-del-problema)  
2. [Objetivo de Negocio](#2-objetivo-de-negocio)  
3. [Stakeholders e Impacto](#3-stakeholders-e-impacto)  
4. [Métricas de Éxito](#4-métricas-de-éxito)  
5. [Análisis de Impacto de Errores](#5-análisis-de-impacto-de-errores)  
6. [Descripción del Dataset](#6-descripción-del-dataset)  
7. [Hipótesis Iniciales](#7-hipótesis-iniciales)  
8. [Supuestos y Limitaciones](#8-supuestos-y-limitaciones)  
9. [Alcance del Proyecto y Próximos Pasos](#9-alcance-del-proyecto-y-próximos-pasos)

---
## 1. Contexto del Problema

La gestión del riesgo crediticio es una de las funciones más críticas en la industria financiera. Cuando un prestatario no cumple con sus obligaciones de deuda — evento conocido como **default o incumplimiento crediticio** — las instituciones financieras absorben pérdidas directas que erosionan su rentabilidad y reservas de capital.

Los enfoques tradicionales de scoring crediticio (scorecards basados en reglas, regresión logística simple) tienen limitaciones conocidas: dependen fuertemente de umbrales estáticos, tienen dificultades con relaciones no lineales entre variables, y frecuentemente no logran adaptarse a cambios en los patrones de comportamiento de pago de los clientes.

**El Machine Learning ofrece una mejora medible** al:
- Capturar interacciones no lineales entre variables financieras.
- Aprovechar señales conductuales (historial de pagos, tendencias de utilización) que los modelos estáticos subutilizan.
- Producir scores de probabilidad calibrados — no solo predicciones binarias — habilitando toma de decisiones segmentada por nivel de riesgo.

Este proyecto simula un pipeline real de modelado de riesgo crediticio para un **banco o fintech**, utilizando un dataset público de 30.000 clientes de tarjetas de crédito de Taiwán (2005). El objetivo es demostrar una solución de ML de extremo a extremo: desde el planteo del problema de negocio hasta un modelo listo para producción.

---
## 2. Objetivo de Negocio

### Objetivo Principal
> **Construir un modelo de clasificación supervisado que prediga la probabilidad de que un cliente de tarjeta de crédito incumpla su pago en el mes siguiente.**

**Variable Target:** `default.payment.next.month`  
- `0` → Sin incumplimiento (el cliente paga a tiempo)
- `1` → Incumplimiento (el cliente no paga)

### Valor para el Negocio

Un modelo bien calibrado permite a la institución:

| Aplicación | Descripción |
|------------|-------------|
| **Mitigación proactiva de riesgo** | Identificar clientes de alto riesgo antes del incumplimiento y activar acciones tempranas de cobranza |
| **Ajuste dinámico de límites de crédito** | Reducir la exposición en clientes con scores de riesgo crecientes |
| **Aprobación automatizada de créditos** | Integrar los scores del modelo en pipelines de underwriting en tiempo real |
| **Provisión de capital** | Mejorar las estimaciones de pérdida esperada (EL) para reportes regulatorios |
| **Segmentación de clientes** | Separar niveles de riesgo bajo/medio/alto para ofertas de productos personalizados |

### Preguntas Analíticas a Responder

1. ¿Qué variables tienen mayor poder predictivo sobre el incumplimiento crediticio?
2. ¿Los patrones de pago conductuales (PAY_0–PAY_6) superan en poder predictivo a las variables demográficas?
3. ¿Cuál es el umbral de probabilidad óptimo para maximizar el valor de negocio dada la asimetría de costos entre falsos negativos y falsos positivos?
4. ¿Qué tan estable e interpretable es el modelo para uso regulatorio y operativo?

---
## 3. Stakeholders e Impacto

| Stakeholder | Rol | Interés en el Modelo |
|-------------|-----|----------------------|
| **Equipo de Gestión de Riesgos** | Usuario principal | Usar scores para activar alertas y ajustar políticas de crédito |
| **Operaciones de Crédito** | Usuario operativo | Integrar el output del modelo en flujos diarios de aprobación |
| **Finanzas / CFO** | Sponsor ejecutivo | Reducir ratios de cartera vencida (NPL), mejorar precisión del provisionamiento |
| **Compliance y Regulatorio** | Supervisión | Garantizar equidad del modelo, auditabilidad (SHAP / explicabilidad) |
| **Clientes** | Parte afectada | Las decisiones crediticias impactan su acceso a productos financieros |
| **Equipo de Data Science** | Dueños del modelo | Construir, monitorear y reentrenar el modelo en el tiempo |

> **Nota de equidad (Fairness):** Las variables demográficas (`SEX`, `EDUCATION`, `MARRIAGE`, `AGE`) están presentes en este dataset y serán analizadas con cuidado. Cualquier patrón discriminatorio detectado debe ser documentado y abordado antes de un despliegue en producción.

---
## 4. Métricas de Éxito

Las métricas se seleccionan considerando el **desbalance de clases** presente en el dataset (~22% de tasa de incumplimiento) y el **costo asimétrico** de los errores de predicción en riesgo crediticio.

### Métricas Primarias

| Métrica | Fórmula | Justificación |
|---------|---------|---------------|
| **ROC-AUC** | Área bajo la curva ROC | Medida de poder discriminatorio independiente del umbral. Estándar de la industria para scoring crediticio. |
| **Recall (Sensibilidad)** | VP / (VP + FN) | Minimizar los falsos negativos es crítico — no detectar un deudor tiene alto costo financiero. |
| **Precisión** | VP / (VP + FP) | Controla la tasa de falsos positivos — sobre-identificar clientes riesgosos afecta la experiencia del cliente. |
| **F1-Score** | 2 · (P · R) / (P + R) | Media armónica que balancea precisión y recall bajo desbalance de clases. |

### Métricas Secundarias

| Métrica | Justificación |
|---------|---------------|
| **PR-AUC** (AUC Precisión-Recall) | Más informativa que ROC-AUC bajo desbalance de clases significativo. |
| **Brier Score** | Mide la calibración de probabilidades — esencial para aplicaciones segmentadas por nivel de riesgo. |
| **Estadístico KS** | Común en validación de scorecards bancarios; mide la separación máxima entre clases. |

### Benchmarks Objetivo *(a validar en la fase de modelado)*

| Métrica | Mínimo Aceptable | Objetivo |
|---------|-----------------|----------|
| ROC-AUC | 0.75 | ≥ 0.82 |
| Recall (clase default) | 0.60 | ≥ 0.70 |
| F1-Score (clase default) | 0.50 | ≥ 0.60 |

> **Nota:** La Accuracy se excluye intencionalmente como métrica principal debido al desbalance de clases. Un modelo naive que predice que ningún cliente incumplirá alcanza ~78% de accuracy — un baseline engañoso.

---
## 5. Análisis de Impacto de Errores

Entender el costo de cada tipo de error es fundamental para elegir la estrategia de optimización correcta.

### Marco de la Matriz de Confusión

```
                          Predicho: Sin Incumplimiento    Predicho: Incumplimiento
Real: Sin Incumplimiento      Verdadero Negativo (VN)      Falso Positivo (FP)
Real: Incumplimiento          Falso Negativo (FN)          Verdadero Positivo (VP)
```

### Análisis de Costos

| Tipo de Error | Consecuencia para el Negocio | Severidad |
|---------------|-----------------------------|-----------|
| **Falso Negativo (FN)** | El cliente incumple pero no fue identificado → la institución absorbe la pérdida crediticia completa | 🔴 **ALTO** |
| **Falso Positivo (FP)** | El cliente es marcado como riesgoso pero habría pagado → reducción innecesaria del límite, posible churn, daño reputacional | 🟡 **MEDIO** |

### Implicancia Estratégica

> **Los Falsos Negativos son significativamente más costosos que los Falsos Positivos** en este contexto.  
> Esto impulsa la decisión de **priorizar el Recall** sobre la Precisión y de considerar un **umbral de clasificación más bajo** (por ejemplo, 0.3–0.4 en lugar de 0.5) para capturar más defaults reales — aceptando un incremento controlado de falsos positivos como trade-off.  
> El umbral óptimo se determinará empíricamente durante la fase de modelado mediante análisis costo-sensible.

---
## 6. Descripción del Dataset

**Fuente:** [UCI Machine Learning Repository — Default of Credit Card Clients](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients)  
**Cita:** Yeh, I. C., & Lien, C. H. (2009). *The comparisons of data mining techniques for the predictive accuracy of probability of default of credit card clients.* Expert Systems with Applications, 36(2), 2473–2480.

### Características Generales

| Propiedad | Valor |
|-----------|-------|
| Observaciones | 30.000 |
| Variables | 23 + 1 target |
| Período | Abril–Septiembre 2005 |
| Geografía | Taiwán |
| Tipo de tarea | Clasificación binaria |
| Balance de clases | ~77.9% Sin Default / ~22.1% Default |

### Diccionario de Variables

#### Variables Demográficas
| Variable | Tipo | Descripción | Notas |
|----------|------|-------------|-------|
| `SEX` | Categórica | 1 = Masculino, 2 = Femenino | — |
| `EDUCATION` | Categórica | 1=Posgrado, 2=Universidad, 3=Secundaria, 4=Otros, 5/6=Desconocido | Categorías 5 y 6 no documentadas — a investigar en EDA |
| `MARRIAGE` | Categórica | 1=Casado, 2=Soltero, 3=Otros, 0=Desconocido | Categoría 0 no documentada |
| `AGE` | Numérica | Edad en años | — |

#### Variables Financieras
| Variable | Tipo | Descripción |
|----------|------|-------------|
| `LIMIT_BAL` | Numérica | Límite de crédito (dólar taiwanés) — individual + suplementario |
| `BILL_AMT1–6` | Numérica | Saldo del estado de cuenta para los meses t-1 a t-6 (más reciente = AMT1) |
| `PAY_AMT1–6` | Numérica | Monto pagado para los meses t-1 a t-6 |

#### Variables de Historial de Pagos
| Variable | Tipo | Descripción | Valores |
|----------|------|-------------|--------|
| `PAY_0` | Categórica | Estado de pago en Septiembre 2005 (más reciente) | -2=Sin consumo, -1=Pagado en su totalidad, 0=Crédito revolvente, 1–9=Meses de retraso |
| `PAY_2–6` | Categórica | Estado de pago para meses anteriores | Misma escala |

> **Nota importante sobre la codificación de PAY_X:** Los valores -2 y 0 no están claramente documentados en el paper original. Su significado real será investigado y validado durante el EDA mediante análisis distribucional.

#### Variable Target
| Variable | Tipo | Descripción |
|----------|------|-------------|
| `default.payment.next.month` | Binaria | 1 = Incumplimiento en Octubre 2005, 0 = Sin incumplimiento |

In [12]:
# Verificación inicial del dataset — se expandirá en 02_EDA
import pandas as pd
import numpy as np

# Cargar dataset (ajustar ruta según sea necesario)
df = pd.read_excel('../data/raw/default of credit card clients.xls', header=1)

print(f"Dimensiones del dataset: {df.shape}")
print(f"\nDistribución del target:")
print(df['default payment next month'].value_counts(normalize=True).rename({0:'Sin Default', 1:'Default'}).map('{:.1%}'.format))
print(f"\nValores nulos: {df.isnull().sum().sum()}")
print(f"\nFilas duplicadas: {df.duplicated().sum()}")
df.head(5)

Dimensiones del dataset: (30000, 25)

Distribución del target:
default payment next month
Sin Default    77.9%
Default        22.1%
Name: proportion, dtype: str

Valores nulos: 0

Filas duplicadas: 0


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


---
## 7. Hipótesis Iniciales

Las siguientes hipótesis se formulan a partir del conocimiento del dominio y revisión de literatura. Serán validadas o rechazadas durante el EDA mediante análisis estadístico y visual.

| # | Hipótesis | Variable(s) | Dirección Esperada | Prioridad |
|---|-----------|-------------|--------------------|-----------|
| H1 | Los retrasos de pago recientes son el predictor más fuerte de incumplimiento | `PAY_0`, `PAY_2` | Mayor retraso → Mayor probabilidad de default | 🔴 Alta |
| H2 | Alta utilización del crédito (BILL_AMT / LIMIT_BAL) se correlaciona con mayor riesgo | `BILL_AMT1–6`, `LIMIT_BAL` | Mayor utilización → Mayor riesgo | 🔴 Alta |
| H3 | Clientes que pagan consistentemente menos de lo facturado tienen mayor riesgo | `BILL_AMT1–6`, `PAY_AMT1–6` | Brecha de pago creciente → Mayor riesgo | 🔴 Alta |
| H4 | Límites de crédito más bajos se asocian con mayores tasas de incumplimiento | `LIMIT_BAL` | Límite menor → Mayor default | 🟡 Media |
| H5 | Clientes más jóvenes tienen mayores tasas de incumplimiento | `AGE` | Correlación negativa con default | 🟡 Media |
| H6 | Menor nivel educativo se asocia con mayor incumplimiento | `EDUCATION` | Menor educación → Mayor default | 🟡 Media |
| H7 | El estado civil tiene poder predictivo limitado | `MARRIAGE` | Señal débil esperada | 🟢 Baja |
| H8 | El género no es un predictor fuerte (y debe monitorearse por sesgo) | `SEX` | Diferencia mínima o no significativa | 🟢 Baja / Fairness |

> Todas las hipótesis serán evaluadas sistemáticamente en `02_EDA.ipynb` mediante análisis univariado, bivariado y pruebas de significancia estadística.

---
## 8. Supuestos y Limitaciones

### Supuestos
- El dataset es representativo de la población objetivo para el período estudiado (Abril–Septiembre 2005).
- La variable target está correctamente etiquetada y libre de errores sistemáticos.
- El comportamiento histórico de pagos es un proxy válido para el riesgo de incumplimiento futuro.
- El modelo está pensado para un contexto de **scoring estático** (scoring mensual por lotes), no en tiempo real.

### Limitaciones

| Limitación | Impacto Potencial | Mitigación |
|------------|------------------|------------|
| **Alcance temporal:** datos de un único año (2005) | El modelo puede no generalizar a diferentes ciclos económicos | Documentar claramente; recomendar reentrenamiento con datos recientes |
| **Especificidad geográfica:** mercado crediticio de Taiwán | Los patrones de comportamiento pueden diferir en otros mercados | Validar con datos locales antes de desplegar en otro contexto |
| **Categorías de variables no documentadas** (EDUCATION 5/6, MARRIAGE 0, PAY_X -2/0) | Riesgo de mala interpretación o fuga de información | Investigadas y documentadas en el EDA |
| **Sin variable de ingresos disponible** | Los ingresos son un driver clave del riesgo crediticio; su ausencia es una brecha estructural | Aproximar mediante ratios LIMIT_BAL / montos de pago como proxies |
| **Desbalance de clases (~22% default)** | Los modelos naive tendrán bajo desempeño en la clase minoritaria | Aplicar estrategias de resampling y optimización de umbral |
| **Snapshot estático:** sin modelado de series de tiempo | Las tendencias de comportamiento dinámico no se capturan completamente | Usar la ventana de 6 meses de variables como proxy parcial |

---
## 9. Alcance del Proyecto y Próximos Pasos

### Qué cubre este proyecto
- ✅ Planteo del problema de negocio y definición del objetivo
- ✅ Análisis Exploratorio de Datos (EDA) con validación de hipótesis
- ✅ Preprocesamiento de datos y feature engineering
- ✅ Selección, entrenamiento y evaluación de modelos (múltiples algoritmos)
- ✅ Interpretabilidad del modelo (valores SHAP)
- ✅ Recomendaciones de negocio a partir del output del modelo

### Próximos Pasos Inmediatos

| Paso | Notebook | Descripción |
|------|----------|-------------|
| 1 | `02_EDA.ipynb` | Cargar datos, validar calidad, explorar distribuciones, testear hipótesis |
| 2 | `03_preprocessing.ipynb` | Limpieza, encoding, tratar categorías no documentadas, crear features (tasa de utilización, tendencia de pagos), y escalar |
| 4 | `04_feature_engineering.ipynb` | Creación de variables nuevas derivadas del comportamiento de pago |
| 5 | `05_modeling.ipynb` | Entrenar Regresión Logística, Random Forest, XGBoost; ajustar umbrales; evaluar con suite completa de métricas |
| 6 | `06_conclusions.ipynb` | Traducir los hallazgos del modelo en recomendaciones de negocio accionables |

---

*Fin del Entendimiento del Negocio — continuar con `02_EDA.ipynb`*